# Barnett & Onnela method: sampling and bridging

History of each gap = the days outside its test fold (same cross-validation as the model).

**Scenarios** (Section 4.8):
- `wp`: gaps with movement, with waypoints. Each segment between visible points is simulated and bridged separately.
- `nowp`: gaps with movement, without waypoints.
- `static`: gaps without movement.


In [ ]:
import os
from pathlib import Path
import numpy as np, pandas as pd
from scipy.special import gammaln
from pyproj import Transformer

ROOT = Path.cwd().parent                             
INPUT_CSV = ROOT / "final_dataset_thess9.csv"
PRED_DIR = ROOT / "noaug_w0_lr1e3" / "summary"        

# scenario: (gap file, gaps with movement or not, output folder)
SCENARIOS = {"wp":     ("wp",   1, ROOT / "noaug_w0_lr1e3" / "runs_barnett_wp"),
             "nowp":   ("nowp", 1, ROOT / "noaug_w0_lr1e3" / "runs_barnett"),
             "static": ("nowp", 0, ROOT / "noaug_w0_lr1e3" / "runs_barnett_nonmoving")}
FOLDS = [int(os.environ["FOLD"])] if "FOLD" in os.environ else [1, 2, 3, 4]   # on a cluster: one job per fold

NU, C_GEO, C_TIME, S_DAY = 30.0, 0.02, 0.9, 24.0     # GLC kernel (c_geo in 1/m, c_time in 1/h)
N_RUNS = 100                                          # simulations per gap
SAFETY = 0.5                                         
SEED = 0
SAVE_EVERY = 20                                     

N_FOLDS = 4
DAY_SHIFT = pd.Timedelta(hours=5)                     # the day changes at 05:00

th = pd.read_csv(INPUT_CSV)
th["day"] = (pd.to_datetime(th.real_time) - DAY_SHIFT).dt.strftime("%Y-%m-%d")
FOLD_DAYS = {k + 1: set(d) for k, d in enumerate(np.array_split(np.sort(th.day.unique()), N_FOLDS))}
POS = pd.Series(np.arange(len(th)), index=th.real_time.values)        # timestamp → row
TX, TY, TT, TB = th.x.to_numpy(), th.y.to_numpy(), th.t_j.to_numpy(), th.B.to_numpy()
TO_LL = Transformer.from_crs("epsg:2100", "epsg:4326", always_xy=True)

## 1. History and gaps

- **History:** all events with B ≠ 2 and known displacement, except the days of the test fold. Flights: B = 1, pauses: B = 0.
- **Gaps:** the model's gaps for the scenario. Each gap is split into segments between the visible points (endpoints and waypoints).

In [ ]:
def history(F):
    ok = (th.B != 2) & th[["dt", "dx", "dy"]].notna().all(axis=1) & ~th.day.isin(FOLD_DAYS[F])
    h = th[ok]
    B = h.B.to_numpy()
    return dict(x=h.x.to_numpy(), y=h.y.to_numpy(), t=h.t_j.to_numpy(), B=B,
                dx=h.dx.to_numpy(), dy=h.dy.to_numpy(), dt=h.dt.to_numpy(),
                flights=np.where(B == 1)[0], pauses=np.where(B == 0)[0],
                denom=B[:-1] == 1, num=(B[:-1] == 1) & (B[1:] == 1))


def load_gaps(scen, F):
    file, moving, _ = SCENARIOS[scen]
    P = pd.read_csv(PRED_DIR / f"pred_fold{F}_{file}.csv")
    P = P[P.moving == moving]
    gaps = []
    for win_id, w in P.groupby("win_id", sort=False):
        w = w.sort_values("step_index").reset_index(drop=True)
        vis = [int(POS[r]) for r in w[w.point_type.isin(["Anchor", "Waypoint"])].real_time]
        hid = [int(POS[r]) for r in w[w.is_hidden == 1].real_time]
        assert hid == [k for k in range(vis[0] + 1, vis[-1]) if k not in vis], win_id
        assert th.day.iloc[vis[0]] in FOLD_DAYS[F], win_id               
        gaps.append({"win_id": win_id, "pred": w, "vis": vis})
    return gaps

## 2. Sampling


In [ ]:
def t_pdf(x, df=NU):
    """Student-t density."""
    c = np.exp(gammaln((df + 1) / 2) - gammaln(df / 2)) / np.sqrt(df * np.pi)
    return c * (1 + np.asarray(x) ** 2 / df) ** (-(df + 1) / 2)


def weights(H, x, y, t):
    """GLC weights of all history events at position (x, y) and time t."""
    w_geo = t_pdf(C_GEO * np.sqrt((x - H["x"]) ** 2 + (y - H["y"]) ** 2))
    m = np.abs(t - H["t"]) % S_DAY
    return w_geo * t_pdf(C_TIME * np.minimum(m, S_DAY - m))


def simulate(H, a_i, b_i, last_B0):
    """Mean position of N_RUNS simulations at the times of the masked points of segment a_i..b_i."""
    t0, t1, t_hid = TT[a_i], TT[b_i], TT[a_i + 1:b_i]
    np.random.seed(SEED + 1000 * a_i + (b_i - a_i))                   # same result on every run
    out = []
    for _ in range(N_RUNS):
        t, pos, last_B = t0, np.array([TX[a_i], TY[a_i]]), last_B0
        run_t, run_x, run_y = [t0], [pos[0]], [pos[1]]
        while True:
            w = weights(H, pos[0], pos[1], t)
            den = np.sum(w[1:][H["denom"]])
            psi = np.sum(w[1:][H["num"]]) / den if den > 0 else 0.0
            if np.random.rand() < last_B * psi + (1 - last_B) and len(H["flights"]) > 0:
                idx, last_B = H["flights"], 1
            elif len(H["pauses"]) > 0:
                idx, last_B = H["pauses"], 0
            else:
                break
            wk = w[idx]
            p = wk / wk.sum() if wk.sum() > 0 else np.ones_like(wk) / len(wk)
            k = np.random.choice(idx, p=p)
            if t + H["dt"][k] >= t1:                                     # the event goes past the end
                break
            t += H["dt"][k]
            pos += np.array([H["dx"][k], H["dy"][k]])
            run_t.append(t); run_x.append(pos[0]); run_y.append(pos[1])
        out.append(np.c_[np.interp(t_hid, run_t, run_x), np.interp(t_hid, run_t, run_y)])
    return np.mean(out, axis=0)

## 3. Bridging

- **S(τ₁):** from the last masked point (S_N, t_N), the fraction r of the expected flight Δ̄ is added.
- **Correction:** Xᵢ = Sᵢ + (tᵢ − τ₀)/(τ₁ − τ₀) × [b − S(τ₁)].

In [ ]:
def s_tau1(H, S, t_hid, t1, a, b):
    """Estimate of the simulated position at time τ1"""
    f = H["flights"]
    w = weights({"x": H["x"][f], "y": H["y"][f], "t": H["t"][f]}, S[-1, 0], S[-1, 1], t_hid[-1])
    if w.sum() == 0:
        q, dtq = np.zeros(2), 1.0
    else:
        q = np.array([np.average(H["dx"][f], weights=w), np.average(H["dy"][f], weights=w)])
        dtq = np.average(H["dt"][f], weights=w)
    if np.hypot(*q) > SAFETY * np.hypot(*(b - a)):
        return S[-1]
    return S[-1] + min(max((t1 - t_hid[-1]) / dtq, 0.0), 1.0) * q


def bridge(H, S, a_i, b_i):
    """Modified bridging of the positions S"""
    t0, t1, t_hid = TT[a_i], TT[b_i], TT[a_i + 1:b_i]
    a, b = np.array([TX[a_i], TY[a_i]]), np.array([TX[b_i], TY[b_i]])
    w = ((t_hid - t0) / (t1 - t0))[:, None]
    return S + w * (b - s_tau1(H, S, t_hid, t1, a, b))

## Degrees of freedom of the kernel (Section 2.6)


In [ ]:
a_i = int(POS["2025-11-26 17:27:30"]); b_i = int(POS["2025-11-26 17:51:28"])
ok = (th.B == 1) & th[["dt", "dx", "dy"]].notna().all(axis=1) & ~th.index.isin(range(a_i, b_i))
h = th[ok]
m = np.abs(TT[a_i] - h.t_j.to_numpy()) % S_DAY
for nu in [1, 30]:
    w = t_pdf(C_GEO * np.hypot(TX[a_i] - h.x.to_numpy(), TY[a_i] - h.y.to_numpy()), nu) * t_pdf(C_TIME * np.minimum(m, S_DAY - m), nu)
    print(f"ν = {nu}: the 50 events with the largest weight take {np.sort(w)[::-1][:50].sum() / w.sum():.1%} of the probability")

## 4. Execution

For each gap: sampling and bridging per segment.

In [ ]:
def path_dist(px, py, poly):
    """Distance of the point from the true route."""
    a, v = poly[:-1], poly[1:] - poly[:-1]
    t = np.clip(((np.array([px, py]) - a) * v).sum(1) / np.maximum((v ** 2).sum(1), 1e-9), 0.0, 1.0)
    q = a + t[:, None] * v
    return float(np.hypot(px - q[:, 0], py - q[:, 1]).min())


def pred_rows(g, X, tag):
    d = g["pred"].copy()
    h = d.is_hidden.to_numpy() == 1
    ia, ib = g["vis"][0], g["vis"][-1]
    tx, ty = TX[ia:ib + 1], TY[ia:ib + 1]
    hm = ~np.isin(np.arange(ia, ib + 1), g["vis"])
    px, py = tx.copy(), ty.copy()
    px[hm], py[hm] = X[:, 0], X[:, 1]
    lon, lat = TO_LL.transform(X[:, 0], X[:, 1])
    err = np.hypot(X[:, 0] - tx[hm], X[:, 1] - ty[hm])
    poly = np.c_[tx, ty]
    d.loc[h, "lat"], d.loc[h, "lon"], d.loc[h, "err_m"] = lat, lon, err
    d.loc[h, "path_err_m"] = [path_dist(u, v, poly) for u, v in X]
    d.loc[h, "err_norm"] = err / d.loc[h, "gap_step_m"].to_numpy()
    st = np.hypot(np.diff(px), np.diff(py))
    d["step_cv_pred"] = round(float(st.std() / st.mean()), 3) if st.mean() > 0 else 0.0
    d["scenario"] = tag
    return d


def save(rows, path):
    if rows:
        pd.concat(rows).to_csv(path, mode="a", header=not path.exists(), index=False)
        rows.clear()


def run(scen, F, only=None):
    out = SCENARIOS[scen][2] / f"fold{F}"
    out.mkdir(parents=True, exist_ok=True)
    f_raw, f_br = out / f"MC100_pred_fold{F}.csv", out / f"MC100_BSTAR_pred_fold{F}.csv"
    done = set(pd.read_csv(f_br, usecols=["win_id"]).win_id) if f_br.exists() else set()
    H = history(F)
    gaps = [g for g in load_gaps(scen, F) if g["win_id"] not in done and (only is None or g["win_id"] in only)]
    raw, br = [], []
    for n, g in enumerate(gaps, 1):
        before = np.where(TB[:g["vis"][0]] != 2)[0]                    # state before the whole gap
        last_B0 = 1 if len(before) == 0 else int(TB[before[np.argmax(TT[before])]] == 1)
        S, X = [], []
        for a_i, b_i in zip(g["vis"][:-1], g["vis"][1:]):
            if b_i - a_i > 1:
                S.append(simulate(H, a_i, b_i, last_B0))
                X.append(bridge(H, S[-1], a_i, b_i))
        raw.append(pred_rows(g, np.vstack(S), f"MC100_{scen}"))
        br.append(pred_rows(g, np.vstack(X), "MC100_BSTAR"))
        if n % SAVE_EVERY == 0:
            save(raw, f_raw); save(br, f_br)
    save(raw, f_raw); save(br, f_br)
    print(f"{scen} fold {F}: {len(gaps)} gaps → {f_br}")


for scen in SCENARIOS:
    for F in FOLDS:
        run(scen, F)